# 第11回 解答

## UCB

In [ ]:
import math
import random
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from bandit import *

class UcbStrategy:
    ''' UCB：X_j + c sqrt(2 log n / n_j) が最大の台を引く '''
    c = 1.0     # クラス変数として定数 c を持たせ，後で変えて実験する
    def __init__(self, n):
        self.n = n
        self.reset()
    def reset(self):
        self.t = 0                  # 今までに引いた回数
        self.count = [0]*self.n     # それぞれの台を引いた回数
        self.win = [0]*self.n       # それぞれの台で当たった回数
    def learn(self, k, result):
        self.t += 1
        self.count[k] += 1
        self.win[k] += result
    def choice(self):
        for k in range(self.n):     # まだ引いていない台があれば，それを引く
            if self.count[k] == 0:
                return k
        best, bestv = 0, -1
        for k in range(self.n):
            v = self.win[k]/self.count[k] + self.c * math.sqrt(2 * math.log(self.t) / self.count[k])
            if v > bestv:
                best, bestv = k, v
        return best

ret = MultiMABTester(200, [UcbStrategy, TsuikiStrategy, RandomStrategy])
print([round(float(x.mean()), 3) for x in ret], "(UCB c=1, Tsuiki, Random, MAX)")

In [ ]:
# c の値を変えて比べる（1 ゲーム 300 回）
for c in [0.05, 0.1, 0.2, 0.3, 0.5, 1.0]:
    UcbStrategy.c = c
    ret = MultiMABTester(200, [UcbStrategy])
    print(f"c = {c}: {ret[0].mean():.3f}")

**考察**

教科書どおりの $c = 1$（UCB1）では，300 回という短いゲームでは探索（まだよく分からない台を試すこと）をしすぎて，Tsuiki 戦略よりも成績が悪くなります。$c$ を 0.05〜0.1 くらいに小さくすると，成績は 0.87 程度まで上がります。UCB1 が「よい」というのは，引く回数が十分大きい時の理論的な保証（後悔の増え方が $\log n$ 程度）であって，300 回のような短いゲームで一番よいとは限りません。引く回数を大きくすると，$c = 1$ の成績は次第に上がります。

In [ ]:
# 引く回数を大きくした時（時間がかかるのでゲーム数は少なめ）
for c in [0.1, 1.0]:
    UcbStrategy.c = c
    ret = MultiMABTester(30, [UcbStrategy, TsuikiStrategy], numplay=3000)
    print(f"3000 回, c = {c}: UCB {ret[0].mean():.3f}, Tsuiki {ret[1].mean():.3f}, MAX {ret[2].mean():.3f}")
UcbStrategy.c = 0.1

## 様々な確率密度関数に従う乱数

In [ ]:
# 練習問題（平均が 0.5，分散が 0.2 のガウス分布）
# 平均 0，分散 1 の乱数 z に対し，mu + sigma * z は平均 mu，分散 sigma^2 のガウス分布に従う
mu, var = 0.5, 0.2
sigma = math.sqrt(var)
y = mu + sigma * np.random.randn(100000)
fig, ax = plt.subplots()
ax.hist(y, bins=100, density=True)
x = np.linspace(-1.5, 2.5, 200)
ax.plot(x, np.exp(-(x-mu)**2/(2*var)) / np.sqrt(2*np.pi*var))
plt.show()
print(y.mean(), y.var())

In [ ]:
# 余裕があれば：Box-Muller 法
# u1, u2 が (0,1] の一様乱数の時，sqrt(-2 log u1) cos(2 pi u2) は平均 0，分散 1 のガウス分布に従う
def boxmuller():
    u1 = 1 - random.random()
    u2 = random.random()
    return math.sqrt(-2*math.log(u1)) * math.cos(2*math.pi*u2)

y = [boxmuller() for _ in range(100000)]
print(np.mean(y), np.var(y))

**一様分布の平均と分散** $[a,b]$ の一様分布の確率密度関数は $f(x) = 1/(b-a)$ なので，
$$
\mu = \int_a^b \frac{x}{b-a} dx = \frac{a+b}{2}, \qquad
\sigma^2 = \int_a^b \frac{(x-\mu)^2}{b-a} dx = \frac{1}{b-a}\left[\frac{(x-\mu)^3}{3}\right]_a^b = \frac{(b-a)^2}{12}.
$$

## 大数の法則と中心極限定理

In [ ]:
# 練習問題（[-1, 1] の一様乱数 n 個の平均を，100 回求める）
def average(n):
    return sum(random.uniform(-1, 1) for _ in range(n)) / n

for n in [1, 10, 100, 1000]:
    a = [average(n) for _ in range(100)]
    print(f"n = {n:4}: 平均 {np.mean(a):+.4f}, 分散 {np.var(a):.5f}  （理論値 {1/3/n:.5f}）")
# [-1, 1] の一様分布の分散は 1/3 なので，n 個の平均の分散は 1/(3n) になる

In [ ]:
# 練習問題（avrandom：n 個の一様乱数の平均を返す乱数生成器）
class avrandom:
    def __init__(self, n):
        self.n = n
    def next(self):
        return sum(random.uniform(-1, 1) for _ in range(self.n)) / self.n

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, n in zip(axes, [1, 2, 5, 30]):
    y = [avrandom(n).next() for _ in range(20000)]
    ax.hist(y, bins=80, density=True)
    s = math.sqrt(1/3/n)          # 平均 0，分散 1/(3n) のガウス分布と重ねる
    x = np.linspace(-1, 1, 200)
    ax.plot(x, np.exp(-x**2/(2*s*s)) / (s*math.sqrt(2*math.pi)))
    ax.set_title(f"n = {n}")
plt.show()

In [ ]:
# いろいろな乱数の和：例えば指数分布（平均 1，分散 1）の n 個の平均
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n in zip(axes, [1, 5, 50]):
    y = np.random.exponential(1.0, size=(20000, n)).mean(axis=1)
    ax.hist(y, bins=80, density=True)
    s = math.sqrt(1/n)
    x = np.linspace(max(0, 1-4*s), 1+4*s, 200)
    ax.plot(x, np.exp(-(x-1)**2/(2*s*s)) / (s*math.sqrt(2*math.pi)))
    ax.set_title(f"n = {n}")
plt.show()

## 任意の確率密度関数に従う乱数

In [ ]:
# 練習問題（指数分布に従う乱数発生器）
class Exponential:
    def __init__(self, lam=1.0):
        self.lam = lam
    def next(self):
        y = random.random()                 # [0, 1) の一様乱数
        return -math.log(1 - y) / self.lam  # 累積分布関数の逆関数

e = Exponential(2.0)
y = [e.next() for _ in range(100000)]
print("平均", np.mean(y), "（理論値 1/λ =", 1/2.0, "）")
fig, ax = plt.subplots()
ax.hist(y, bins=100, density=True)
x = np.linspace(0, 5, 200)
ax.plot(x, 2.0*np.exp(-2.0*x))
plt.show()

In [ ]:
# 任意の確率密度関数に従う乱数（累積分布関数を配列で近似し，逆関数は2分探索で求める）
class InverseSampler:
    def __init__(self, f, xmin, xmax, k=2**16):
        self.x = np.linspace(xmin, xmax, k)
        cdf = np.cumsum(f(self.x))
        self.cdf = cdf / cdf[-1]           # 正規化して，最後が 1 になるようにする
    def next(self):
        y = random.random()
        i = np.searchsorted(self.cdf, y)   # cdf[i] >= y となる最小の i（2分探索）
        return self.x[i]

s = InverseSampler(lambda x: x*x*(1-x), 0, 1)     # x^2 (1-x) に比例する分布（ベータ分布 a=3, b=2）
y = [s.next() for _ in range(50000)]
fig, ax = plt.subplots()
ax.hist(y, bins=100, density=True)
x = np.linspace(0, 1, 200)
ax.plot(x, 12*x*x*(1-x))
plt.show()

## ベイズの定理

In [ ]:
# 練習問題（モンティ・ホール問題のシミュレーション）
def monty(change, n=100000):
    win = 0
    for _ in range(n):
        atari = random.randrange(3)
        choice = random.randrange(3)
        # 司会者は，解答者が選んでいない，はずれの箱を開ける
        opened = random.choice([b for b in range(3) if b != choice and b != atari])
        if change:
            choice = [b for b in range(3) if b != choice and b != opened][0]
        win += (choice == atari)
    return win / n

print("変えない:", monty(False), " 変える:", monty(True))

In [ ]:
# 司会者が，はずれの箱を知らずにランダムに開け，たまたまはずれだった場合
def monty_random(change, n=100000):
    win = 0
    count = 0
    for _ in range(n):
        atari = random.randrange(3)
        choice = random.randrange(3)
        opened = random.choice([b for b in range(3) if b != choice])
        if opened == atari:      # 当たりを開けてしまった場合は数えない
            continue
        count += 1
        if change:
            choice = [b for b in range(3) if b != choice and b != opened][0]
        win += (choice == atari)
    return win / count

print("変えない:", monty_random(False), " 変える:", monty_random(True))
# この場合は，変えても変えなくても 1/2 になる

## トンプソン抽出

In [ ]:
# 練習問題（Thompson Sampling）
class ThompsonStrategy:
    def __init__(self, n):
        self.n = n
        self.reset()
    def reset(self):
        self.count = [0]*self.n
        self.win = [0]*self.n
    def learn(self, k, result):
        self.count[k] += 1
        self.win[k] += result
    def choice(self):
        # それぞれの台の p の事後分布（ベータ分布）から1つずつサンプリングし，最大の台を引く
        betas = [np.random.beta(self.win[k]+1, self.count[k]-self.win[k]+1) for k in range(self.n)]
        return betas.index(max(betas))

UcbStrategy.c = 0.1
names = ["UCB(c=0.1)", "Thompson", "Tsuiki", "Random", "MAX"]
ret = MultiMABTester(200, [UcbStrategy, ThompsonStrategy, TsuikiStrategy, RandomStrategy])
sorted(zip([round(float(x.mean()), 3) for x in ret], names), reverse=True)

In [ ]:
# 練習問題（発展）：change=True の設定で比べる
names = ["Thompson", "Tsuiki", "MAX"]
ret = MultiMABTester(200, [ThompsonStrategy, TsuikiStrategy], change=True)
sorted(zip([round(float(x.mean()), 3) for x in ret], names), reverse=True)

**考察** 確率が変化する設定では，普通のトンプソン抽出は Tsuiki 戦略とほぼ同じか，負けることもあります。トンプソン抽出は「p は一定」という前提で，最初からの全ての結果を同じ重みで使うので，p が変化すると古い情報に引きずられてしまいます。一方，Tsuiki 戦略は直前の結果しか見ないので，変化についていけます。本文の考え方のように，p が ±0.05 ずつ動くことを事後分布の計算に組み込んだらどうなるかを，次で試してみます。

## 確率の変化まで含めた事後分布（発展の練習問題の解答）

それぞれの台の p の分布を，[0, 1] を 1000 等分した格子の上の配列として持ちます。毎ターン，

1. 引いた台の分布に，当たりなら $p$，外れなら $1-p$ を掛けて正規化する（ベイズ更新），
2. 全ての台の分布を，1/2 ずつ +0.05 と -0.05 にずらして足し合わせる（0 より小さくなる分は 0 に，1 より大きくなる分は 1 にまとめる），

という更新を行います。変化の規則が分かっているので，これで（格子で近似した範囲で）正確な事後分布が得られます。これは，隠れマルコフモデルの「フィルタリング」と呼ばれる計算です。

In [ ]:
G = 1001                      # p の格子 0, 0.001, ..., 1 の点の数
P = np.linspace(0, 1, G)
S = 50                        # 0.05 は，格子 50 個分

def drift(d):
    """ 各行が1台の p の分布である配列 d を，p が 1/2 の確率で ±0.05 動いた後の分布にする """
    up = np.zeros_like(d)
    down = np.zeros_like(d)
    up[:, S:] += d[:, :G-S]               # p -> p + 0.05
    up[:, G-1] += d[:, G-S:].sum(axis=1)  # 1 を超える分は 1 に
    down[:, :G-S] += d[:, S:]             # p -> p - 0.05
    down[:, 0] += d[:, :S].sum(axis=1)    # 0 を下回る分は 0 に
    return 0.5*up + 0.5*down

# 確認：p = 0.02 に集中した分布は，1/2 ずつ p = 0 と p = 0.07 に分かれる
d = np.zeros((1, G))
d[0, 20] = 1.0
d2 = drift(d)
print(d2.sum(), d2[0, 0], d2[0, 70])

In [ ]:
class FilterThompson:
    """ 確率の変化まで含めた事後分布から，トンプソン抽出で引く台を決める """
    def __init__(self, n):
        self.n = n
        self.reset()
    def reset(self):
        self.d = np.ones((self.n, G)) / G     # 最初は一様分布
    def learn(self, k, result):
        if result == 1:
            self.d[k] *= P
        else:
            self.d[k] *= (1 - P)
        self.d[k] /= self.d[k].sum()
        self.d = drift(self.d)                # 毎ターン，全ての台の p が動く
    def choice(self):
        c = self.d.cumsum(axis=1)             # 各台の累積分布関数
        u = np.random.rand(self.n, 1)
        samples = (c < u).sum(axis=1)         # 逆関数法で，各台の分布から1つずつサンプリング
        return int(np.argmax(samples))

class FilterGreedy(FilterThompson):
    """ 同じ事後分布を使い，事後平均が最大の台を引く """
    def choice(self):
        return int(np.argmax(self.d @ P))

In [ ]:
for ch in [True, False]:
    names = ["FilterThompson", "FilterGreedy", "Thompson", "Tsuiki", "MAX"]
    ret = MultiMABTester(200, [FilterThompson, FilterGreedy, ThompsonStrategy, TsuikiStrategy], change=ch)
    print(f"change={ch}:", sorted(zip([round(float(x.mean()), 3) for x in ret], names), reverse=True))

**考察**

意外なことに，確率の変化まで正しく取り入れた事後分布を使っても，トンプソン抽出（FilterThompson）は，Tsuiki 戦略や普通のトンプソン抽出よりも弱くなります。一方，同じ事後分布を使って，事後平均が最大の台を引く戦略（FilterGreedy）は，Tsuiki 戦略よりも強くなります。モデルは正しいのに，使い方によって強さが逆転するのです。

原因は，引いていない台の分布が広がり続けることです。確率が一定の問題では，引くほど事後分布が細くなるので，トンプソン抽出の探索は自然に減っていきます。しかし確率が変化する問題では，しばらく引いていない台の分布は，$t$ ターンで標準偏差が $0.05\sqrt{t}$ 程度まで広がり続けます。引いていない9台の分布から1つずつサンプリングすると，その最大値はほとんどいつも高くなるので，今よい台を引いていても，すぐに他の台に移ってしまいます。つまり，探索が止まらなくなります。

事後平均を使うと，引いていない台の平均はあまり動かないので，探索は，今引いている台の事後平均が下がってきた時にだけ起こります。

確率が変化するバンディット問題（restless bandit）の最適な戦略を求めることは，一般には非常に難しい問題として知られています。「正しい事後分布を持っている」ことと，「それを使ってよい判断をする」ことは別の問題だということが，この実験から分かります。

change=False の設定では，FilterThompson と FilterGreedy は「確率は変化する」という間違ったモデルを使っていることになります。それでも FilterGreedy は普通のトンプソン抽出と同じくらいの成績になるのは，興味深いところです。